In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from pathlib import Path
import time
import math
import copy
import matplotlib.pyplot as plt

PARQUET_DIR = Path.home() / "projects" / "recsys" / "data" / "parquet"
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
torch.manual_seed(42)
np.random.seed(42)
print(f"device: {device}")

ratings = pd.read_parquet(PARQUET_DIR / "ratings_clean.parquet")
cutoff_ts = ratings["timestamp"].quantile(0.9)
train_df_full = ratings[ratings["timestamp"] < cutoff_ts].reset_index(drop=True)
val_df_full   = ratings[ratings["timestamp"] >= cutoff_ts].reset_index(drop=True)

user_to_idx = {u: i for i, u in enumerate(train_df_full["userId"].unique())}
movie_to_idx = {m: i for i, m in enumerate(train_df_full["movieId"].unique())}
n_users = len(user_to_idx)
n_movies = len(movie_to_idx)

def apply_mappings(df):
    df = df.copy()
    df["user_idx"] = df["userId"].map(user_to_idx)
    df["movie_idx"] = df["movieId"].map(movie_to_idx)
    df = df.dropna(subset=["user_idx", "movie_idx"]).reset_index(drop=True)
    df["user_idx"] = df["user_idx"].astype(np.int32)
    df["movie_idx"] = df["movie_idx"].astype(np.int32)
    return df

train_df_full = apply_mappings(train_df_full)
val_df_full   = apply_mappings(val_df_full)

LIKE_THRESHOLD = 4.0
train_pos = train_df_full[train_df_full["rating"] >= LIKE_THRESHOLD].reset_index(drop=True)
val_pos   = val_df_full[val_df_full["rating"] >= LIKE_THRESHOLD].reset_index(drop=True)

print(f"users: {n_users:,} | movies: {n_movies:,}")
print(f"train positives: {len(train_pos):,}")
print(f"val positives: {len(val_pos):,}")

device: mps
users: 150,330 | movies: 45,058
train positives: 11,177,787
val positives: 181,949


In [2]:
class PositivesDataset(Dataset):
    """yields (user_idx, pos_movie_idx). negatives come from the batch itself."""
    def __init__(self, pos_df: pd.DataFrame):
        self.users = torch.from_numpy(pos_df["user_idx"].values.astype(np.int64))
        self.movies = torch.from_numpy(pos_df["movie_idx"].values.astype(np.int64))
    def __len__(self):
        return len(self.users)
    def __getitem__(self, i):
        return self.users[i], self.movies[i]


train_ds = PositivesDataset(train_pos)
print(f"train dataset: {len(train_ds):,} positive pairs")

# sanity check: pull one batch
loader = DataLoader(train_ds, batch_size=4, shuffle=True)
u, m = next(iter(loader))
print(f"\nsample batch shape: users={u.shape}, movies={m.shape}")
print(f"users:  {u.tolist()}")
print(f"movies: {m.tolist()}")

train dataset: 11,177,787 positive pairs

sample batch shape: users=torch.Size([4]), movies=torch.Size([4])
users:  [138676, 61050, 1774, 22538]
movies: [1478, 590, 1932, 647]


In [3]:
class TwoTower(nn.Module):
    def __init__(self, n_users: int, n_movies: int, dim: int = 64):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, dim)
        self.item_emb = nn.Embedding(n_movies, dim)
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
    def encode_user(self, u): return self.user_emb(u)
    def encode_item(self, m): return self.item_emb(m)


EMB_DIM = 64
BATCH_SIZE = 8192
LR = 0.005
WEIGHT_DECAY = 1e-5
N_EPOCHS = 5
TEMPERATURE = 0.07  # scales scores before softmax — higher T = softer distribution

model = TwoTower(n_users, n_movies, dim=EMB_DIM).to(device)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

n_params = sum(p.numel() for p in model.parameters())
print(f"params: {n_params:,}")
print(f"batches per epoch: {len(train_loader):,}")
print(f"in-batch negatives per positive: {BATCH_SIZE - 1}")
print(f"temperature: {TEMPERATURE}")
print()

losses = []
accuracies = []  # how often the diagonal element is the row's argmax

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    epoch_correct = 0
    n_seen = 0
    t0 = time.time()
    
    for u, m in train_loader:
        u, m = u.to(device), m.to(device)
        bsz = u.size(0)
        
        optimizer.zero_grad()
        
        user_vecs = model.encode_user(u)   # (B, dim)
        item_vecs = model.encode_item(m)   # (B, dim)
        
        # all-pairs scores: row i, col j = score of user i on item j
        # the diagonal entries are user i's score on their actual positive
        scores = user_vecs @ item_vecs.T   # (B, B)
        scores = scores / TEMPERATURE
        
        # cross-entropy: each row is a probability distribution over B items
        # the correct label for row i is i (their positive sits on the diagonal)
        labels = torch.arange(bsz, device=device)
        loss = F.cross_entropy(scores, labels)
        
        loss.backward()
        optimizer.step()
        
        # track per-batch accuracy: did diagonal win the row?
        with torch.no_grad():
            preds = scores.argmax(dim=1)
            epoch_correct += (preds == labels).sum().item()
        
        epoch_loss += loss.item() * bsz
        n_seen += bsz
    
    avg_loss = epoch_loss / n_seen
    accuracy = epoch_correct / n_seen
    losses.append(avg_loss)
    accuracies.append(accuracy)
    
    print(f"epoch {epoch}/{N_EPOCHS} | loss: {avg_loss:.4f} | "
          f"top-1 acc (out of {BATCH_SIZE}): {accuracy*100:.1f}% | {time.time()-t0:.1f}s")

print("\ndone")

params: 12,504,832
batches per epoch: 1,364
in-batch negatives per positive: 8191
temperature: 0.07

epoch 1/5 | loss: 8.3735 | top-1 acc (out of 8192): 0.2% | 133.4s
epoch 2/5 | loss: 8.1796 | top-1 acc (out of 8192): 0.3% | 155.1s
epoch 3/5 | loss: 8.1476 | top-1 acc (out of 8192): 0.4% | 149.0s
epoch 4/5 | loss: 8.1333 | top-1 acc (out of 8192): 0.4% | 157.6s
epoch 5/5 | loss: 8.1260 | top-1 acc (out of 8192): 0.4% | 150.3s

done


In [4]:
model = TwoTower(n_users, n_movies, dim=EMB_DIM).to(device)
nn.init.normal_(model.user_emb.weight, std=0.05)
nn.init.normal_(model.item_emb.weight, std=0.05)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

losses = []
accuracies = []

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    epoch_correct = 0
    n_seen = 0
    t0 = time.time()
    
    for u, m in train_loader:
        u, m = u.to(device), m.to(device)
        bsz = u.size(0)
        
        optimizer.zero_grad()
        user_vecs = model.encode_user(u)
        item_vecs = model.encode_item(m)
        scores = (user_vecs @ item_vecs.T) / TEMPERATURE
        labels = torch.arange(bsz, device=device)
        loss = F.cross_entropy(scores, labels)
        loss.backward()
        optimizer.step()
        
        with torch.no_grad():
            preds = scores.argmax(dim=1)
            epoch_correct += (preds == labels).sum().item()
        
        epoch_loss += loss.item() * bsz
        n_seen += bsz
    
    avg_loss = epoch_loss / n_seen
    accuracy = epoch_correct / n_seen
    losses.append(avg_loss)
    accuracies.append(accuracy)
    
    print(f"epoch {epoch}/{N_EPOCHS} | loss: {avg_loss:.4f} | "
          f"top-1 acc: {accuracy*100:.1f}% | {time.time()-t0:.1f}s")

epoch 1/5 | loss: 8.3938 | top-1 acc: 0.2% | 121.7s
epoch 2/5 | loss: 8.1821 | top-1 acc: 0.3% | 156.8s
epoch 3/5 | loss: 8.1481 | top-1 acc: 0.4% | 151.9s
epoch 4/5 | loss: 8.1329 | top-1 acc: 0.4% | 145.8s
epoch 5/5 | loss: 8.1252 | top-1 acc: 0.4% | 133.6s


In [5]:
# fresh model
model = TwoTower(n_users, n_movies, dim=EMB_DIM).to(device)
nn.init.normal_(model.user_emb.weight, std=0.05)
nn.init.normal_(model.item_emb.weight, std=0.05)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

u, m = next(iter(DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)))
u, m = u.to(device), m.to(device)

print("=== before any update ===")
with torch.no_grad():
    uv = model.encode_user(u)
    iv = model.encode_item(m)
    raw_scores = uv @ iv.T
    scaled_scores = raw_scores / TEMPERATURE
    
    print(f"user vec: mean={uv.mean().item():+.4f}, std={uv.std().item():.4f}, norm={uv.norm(dim=1).mean().item():.4f}")
    print(f"item vec: mean={iv.mean().item():+.4f}, std={iv.std().item():.4f}, norm={iv.norm(dim=1).mean().item():.4f}")
    print(f"raw scores:    mean={raw_scores.mean().item():+.4f}, std={raw_scores.std().item():.4f}")
    print(f"scaled scores: mean={scaled_scores.mean().item():+.4f}, std={scaled_scores.std().item():.4f}")
    print(f"diagonal mean: {scaled_scores.diag().mean().item():+.4f}")
    print(f"off-diag mean: {(scaled_scores - torch.diag(scaled_scores.diag())).mean().item():+.4f}")

# one training step
labels = torch.arange(BATCH_SIZE, device=device)
optimizer.zero_grad()
uv = model.encode_user(u)
iv = model.encode_item(m)
scores = (uv @ iv.T) / TEMPERATURE
loss = F.cross_entropy(scores, labels)
print(f"\nloss before step: {loss.item():.4f}")
loss.backward()

# inspect gradient magnitudes
ug_norm = model.user_emb.weight.grad.norm().item()
ig_norm = model.item_emb.weight.grad.norm().item()
print(f"user_emb grad norm: {ug_norm:.6f}")
print(f"item_emb grad norm: {ig_norm:.6f}")

optimizer.step()

print("\n=== after one update ===")
with torch.no_grad():
    uv2 = model.encode_user(u)
    iv2 = model.encode_item(m)
    scores2 = (uv2 @ iv2.T) / TEMPERATURE
    loss2 = F.cross_entropy(scores2, labels)
    print(f"loss after step: {loss2.item():.4f}")
    print(f"diagonal mean:   {scores2.diag().mean().item():+.4f}")
    print(f"off-diag mean:   {(scores2 - torch.diag(scores2.diag())).mean().item():+.4f}")

=== before any update ===
user vec: mean=+0.0000, std=0.0500, norm=0.3986
item vec: mean=-0.0001, std=0.0499, norm=0.3979
raw scores:    mean=-0.0000, std=0.0200
scaled scores: mean=-0.0000, std=0.2854
diagonal mean: +0.0038
off-diag mean: -0.0000

loss before step: 9.0478
user_emb grad norm: 0.063097
item_emb grad norm: 0.063646

=== after one update ===
loss after step: 8.7821
diagonal mean:   +0.2694
off-diag mean:   -0.0001


In [6]:
TEMPERATURE = 0.02   # was 0.07; sharper softmax → meaningful gradients

model = TwoTower(n_users, n_movies, dim=EMB_DIM).to(device)
nn.init.normal_(model.user_emb.weight, std=0.05)
nn.init.normal_(model.item_emb.weight, std=0.05)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

losses = []
accuracies = []

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    epoch_correct = 0
    n_seen = 0
    t0 = time.time()
    
    for u, m in train_loader:
        u, m = u.to(device), m.to(device)
        bsz = u.size(0)
        optimizer.zero_grad()
        uv = model.encode_user(u)
        iv = model.encode_item(m)
        scores = (uv @ iv.T) / TEMPERATURE
        labels = torch.arange(bsz, device=device)
        loss = F.cross_entropy(scores, labels)
        loss.backward()
        optimizer.step()
        with torch.no_grad():
            preds = scores.argmax(dim=1)
            epoch_correct += (preds == labels).sum().item()
        epoch_loss += loss.item() * bsz
        n_seen += bsz
    
    avg_loss = epoch_loss / n_seen
    accuracy = epoch_correct / n_seen
    losses.append(avg_loss)
    accuracies.append(accuracy)
    
    print(f"epoch {epoch}/{N_EPOCHS} | loss: {avg_loss:.4f} | top-1 acc: {accuracy*100:.1f}% | {time.time()-t0:.1f}s")

epoch 1/5 | loss: 8.5374 | top-1 acc: 0.1% | 121.0s
epoch 2/5 | loss: 8.3630 | top-1 acc: 0.3% | 136.8s
epoch 3/5 | loss: 8.3273 | top-1 acc: 0.3% | 130.3s
epoch 4/5 | loss: 8.3077 | top-1 acc: 0.3% | 134.7s
epoch 5/5 | loss: 8.2981 | top-1 acc: 0.3% | 146.9s


In [7]:
# fresh model, just enough data to test if higher lr makes loss move
TEMPERATURE = 0.07   # back to the standard value
MINI_LR = 0.05       # 10x bigger than before

model = TwoTower(n_users, n_movies, dim=EMB_DIM).to(device)
nn.init.normal_(model.user_emb.weight, std=0.05)
nn.init.normal_(model.item_emb.weight, std=0.05)
optimizer = optim.Adam(model.parameters(), lr=MINI_LR, weight_decay=WEIGHT_DECAY)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

losses_log = []
accs_log = []

# train just 100 batches and log loss every 20
model.train()
t0 = time.time()
for i, (u, m) in enumerate(train_loader):
    if i >= 100:
        break
    u, m = u.to(device), m.to(device)
    bsz = u.size(0)
    optimizer.zero_grad()
    uv = model.encode_user(u)
    iv = model.encode_item(m)
    scores = (uv @ iv.T) / TEMPERATURE
    labels = torch.arange(bsz, device=device)
    loss = F.cross_entropy(scores, labels)
    loss.backward()
    optimizer.step()
    
    with torch.no_grad():
        acc = (scores.argmax(dim=1) == labels).float().mean().item()
    losses_log.append(loss.item())
    accs_log.append(acc)
    
    if (i + 1) % 20 == 0:
        print(f"batch {i+1:>3d} | loss: {loss.item():.4f} | top-1 acc: {acc*100:.1f}%")

print(f"\ntotal time: {time.time()-t0:.1f}s")
print(f"first batch loss: {losses_log[0]:.4f}")
print(f"last batch loss:  {losses_log[-1]:.4f}")
print(f"loss drop:        {losses_log[0] - losses_log[-1]:.4f}")
print(f"first batch acc:  {accs_log[0]*100:.1f}%")
print(f"last batch acc:   {accs_log[-1]*100:.1f}%")

batch  20 | loss: 10.8102 | top-1 acc: 0.0%
batch  40 | loss: 11.3547 | top-1 acc: 0.0%
batch  60 | loss: 11.7550 | top-1 acc: 0.0%
batch  80 | loss: 12.0578 | top-1 acc: 0.0%
batch 100 | loss: 12.4772 | top-1 acc: 0.0%

total time: 9.1s
first batch loss: 9.0528
last batch loss:  12.4772
loss drop:        -3.4245
first batch acc:  0.0%
last batch acc:   0.0%
